# CardiAnatomy: end-to-end BiV volumetric meshing in Google Colab

**Launch this notebook from GitHub in Colab, then run cells from top to bottom.** It connects the merged CardiAnatomy staged runner to the upstream BiV Volumetric Meshing implementation.

This notebook does **not** choose, download, or redistribute your patient dataset. You provide paths to four 4D NIfTI views and the four upstream segmentation-model packages (or an already extracted nnU-Net workspace) in your own Google Drive.

**Outputs and partial results are written to your Google Drive.** Use a private Drive location and a de-identified case identifier. Do not put identifiable patient data in a public repository or share the output folder publicly.

Important limits:
- The upstream maintainers warn that the pipeline may not behave as expected on scans from data sources other than its intended UK Biobank setting. Review outputs against the source scans.
- Segmentation model assets and input datasets have separate access and licensing terms. You are responsible for choosing data and model assets you are authorized to use.
- Surface stages can run without openCARP. Volumetric and UVC/fiber stages use openCARP-dependent tools; the notebook pauses for explicit license review before those stages.
- Passing mesh QC is a geometric screening result, not proof of anatomical accuracy or clinical validity.

## 1. Check the runtime

For segmentation, choose Runtime → Change runtime type → T4 GPU (or another compatible GPU) in Colab. The notebook uses a separate Python 3.11 environment when the active Colab kernel is Python 3.12 or newer because the upstream README warns that nnU-Net may not work with Python 3.12+.

In [ ]:
import os, sys, platform, subprocess, shutil, json, time, hashlib, tarfile, urllib.request
from pathlib import Path
from datetime import datetime, timezone

print("Kernel Python:", sys.version)
print("Platform:", platform.platform())
print("Git:", shutil.which("git"))

## 2. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab to mount Google Drive.") from exc

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
assert DRIVE_ROOT.is_dir(), "Google Drive did not mount at /content/drive/MyDrive"
print("Drive mounted. Keep input and output folders private.")

## 3. Choose your files and run settings

Edit this cell before continuing.

The four inputs must be **4D cine NIfTI files** and correspond to SAX, LAX 2-chamber, LAX 3-chamber, and LAX 4-chamber. Affine axis-code checks can catch some mismatches, but **do not prove that views are registered to each other**.

For model assets, either:
- set MODEL_ASSET_SOURCE to a Drive directory containing all four exact upstream .tar packages, or
- set it to a directory containing an already extracted nnU-Net workspace with nnUNet_raw and its model datasets.

The notebook does not download a dataset or model tar files for you. Obtain them from a source you are authorized to use and review their current terms first.

The default stage list requests segmentation, contours, surface reconstruction and volumetric meshing. If you only want an initial surface pipeline, remove volumetric from STAGES. Add uvc-fiber after volumetric only after reviewing its requirements and terms.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# REQUIRED: replace EDIT_ME with your own private Drive paths.
SAX_PATH     = DRIVE_ROOT / "EDIT_ME" / "case" / "SAX.nii.gz"
LAX_2CH_PATH = DRIVE_ROOT / "EDIT_ME" / "case" / "LAX_2Ch.nii.gz"
LAX_3CH_PATH = DRIVE_ROOT / "EDIT_ME" / "case" / "LAX_3Ch.nii.gz"
LAX_4CH_PATH = DRIVE_ROOT / "EDIT_ME" / "case" / "LAX_4Ch.nii.gz"

# Folder with the four upstream .tar files OR an extracted nnU-Net workspace.
MODEL_ASSET_SOURCE = DRIVE_ROOT / "EDIT_ME" / "biv_model_assets"

DRIVE_RUNS_DIR = DRIVE_ROOT / "CardiAnatomy_BiV_Runs"
CASE_ID = "case_001"   # pseudonymous path-safe ID; do not use patient-identifying information
INSTANCE = 2
TIMEFRAME = None       # None leaves the upstream default / all frames where supported.
UPSTREAM_REF = "v1-public"

# Ordered stage prefix. Use ["segmentation", "contour", "surface"] to stop at surfaces.
STAGES = ["segmentation", "contour", "surface", "volumetric"]
# Optional: append "uvc-fiber" after "volumetric".

PREFER_GPU = True
STAGE_TIMEOUT_SECONDS = 4 * 60 * 60
MINIMUM_POINTS = 500
MINIMUM_TETRAHEDRA = 1000
MINIMUM_TETRA_SHAPE_QUALITY = 0.05

# Explicitly record that you reviewed access/use terms and are authorized.
DATA_TERMS_REVIEWED = False
MODEL_ASSET_TERMS_REVIEWED = False

# Allowed: not_reviewed, academic_noncommercial, commercial_product_development, other.
INTENDED_USE = "not_reviewed"
OPENCARP_TERMS_REVIEWED = False
OPENCARP_COMMERCIAL_LICENSE_CONFIRMED = False
INSTALL_OPENCARP = False
OPENCARP_BIN_DIR = None  # Existing separately installed binaries path, e.g. /content/openCARP/usr/bin.

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_ROOT = DRIVE_RUNS_DIR / f"{CASE_ID}_{RUN_ID}"
MODEL_WORKSPACE = Path("/content") / f"cardianatomy_nnunet_workspace_{RUN_ID}"
print("Run ID:", RUN_ID)
print("Output folder:", OUTPUT_ROOT)
print("Stages:", STAGES)

## 4. Validate the chosen NIfTI views

This reads NIfTI headers, reports shapes, voxel spacing and orientation codes, and calculates SHA-256 hashes. It does not alter source files, resample images or perform registration.

In [ ]:
try:
    import nibabel as nib
    import numpy as np
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "nibabel", "numpy"], check=True)
    import nibabel as nib
    import numpy as np

VIEWS = {
    "SAX": Path(SAX_PATH),
    "LAX_2CH": Path(LAX_2CH_PATH),
    "LAX_3CH": Path(LAX_3CH_PATH),
    "LAX_4CH": Path(LAX_4CH_PATH),
}
if not CASE_ID or not CASE_ID.replace("_", "").replace("-", "").isalnum():
    raise ValueError("CASE_ID must be a simple path-safe alphanumeric identifier.")
if any("EDIT_ME" in path.parts for path in VIEWS.values()):
    raise ValueError("Edit the four input paths before running this cell.")
allowed_stages = ["segmentation", "contour", "surface", "volumetric", "uvc-fiber"]
if not STAGES or STAGES != allowed_stages[:len(STAGES)]:
    raise ValueError("STAGES must be a non-empty sequential prefix: segmentation, contour, surface, volumetric, uvc-fiber.")
if not DATA_TERMS_REVIEWED:
    raise RuntimeError("Review the data access/use terms, then set DATA_TERMS_REVIEWED = True.")
if not MODEL_ASSET_TERMS_REVIEWED:
    raise RuntimeError("Review the model asset access/use terms, then set MODEL_ASSET_TERMS_REVIEWED = True.")

def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_report = {}
for view_name, path in VIEWS.items():
    if not path.is_file():
        raise FileNotFoundError(f"{view_name} file does not exist: {path}")
    img = nib.load(str(path), mmap=True)
    shape = tuple(int(v) for v in img.shape)
    if len(shape) != 4 or any(v < 1 for v in shape):
        raise ValueError(f"{view_name} must be a 4D cine NIfTI; got shape {shape}")
    affine = np.asarray(img.affine, dtype=float)
    if affine.shape != (4, 4) or not np.isfinite(affine).all() or abs(np.linalg.det(affine[:3, :3])) < 1e-12:
        raise ValueError(f"{view_name} has a non-finite or singular affine.")
    spacing = [float(v) for v in nib.affines.voxel_sizes(affine)[:3]]
    axcodes = tuple(nib.orientations.aff2axcodes(affine))
    input_report[view_name] = {
        "path": str(path), "shape": shape, "voxel_spacing_mm": spacing,
        "axis_codes": axcodes, "sha256": sha256_file(path),
    }
    print(f"{view_name}: shape={shape}; spacing_mm={[round(v, 4) for v in spacing]}; "
          f"axis_codes={axcodes}; SHA256={input_report[view_name]['sha256']}")

if len({tuple(row["axis_codes"]) for row in input_report.values()}) != 1:
    raise ValueError("The four views have different affine axis-code conventions. Resolve this explicitly; the notebook will not silently reorient them.")
print("\nHeader checks passed. This does NOT verify view registration, temporal consistency or anatomical coverage.")

## 5. Set up a compatible Python environment and clone the code

This installs dependencies from the upstream project's requirements file and installs CardiAnatomy with its I/O extra. A separate Python 3.11 environment is used when necessary. The install can download several gigabytes, particularly PyTorch and nnU-Net dependencies.

In [ ]:
import os, sys, subprocess, shutil, urllib.request, tarfile
from pathlib import Path

def run_command(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

REPO_ROOT = Path("/content/Virelion-CardiAnatomy")
UPSTREAM_ROOT = Path("/content/biv-volumetric-meshing")
if not REPO_ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "https://github.com/Virelion-Biotech/Virelion-CardiAnatomy.git", REPO_ROOT])
elif not (REPO_ROOT / ".git").exists():
    raise RuntimeError(f"{REPO_ROOT} exists but is not a git checkout. Rename it manually; files will not be overwritten.")
if not UPSTREAM_ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "--branch", UPSTREAM_REF, "https://github.com/cdttk/biv-volumetric-meshing.git", UPSTREAM_ROOT])
elif not (UPSTREAM_ROOT / ".git").exists():
    raise RuntimeError(f"{UPSTREAM_ROOT} exists but is not a git checkout. Rename it manually; files will not be overwritten.")
upstream_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip()
print("Upstream commit:", upstream_sha, "| ref:", UPSTREAM_REF)

ENV_ROOT = Path("/content/cardianatomy-py311")
if sys.version_info < (3, 12):
    if not ENV_ROOT.exists():
        run_command([sys.executable, "-m", "venv", ENV_ROOT])
    ENV_PYTHON = ENV_ROOT / "bin" / "python"
else:
    MICROMAMBA = Path("/content/micromamba")
    if not MICROMAMBA.exists():
        archive = Path("/content/micromamba.tar.bz2")
        urllib.request.urlretrieve("https://micro.mamba.pm/api/micromamba/linux-64/latest", archive)
        with tarfile.open(archive, "r:bz2") as tar:
            member = tar.getmember("bin/micromamba")
            stream = tar.extractfile(member)
            if stream is None:
                raise RuntimeError("Could not extract micromamba executable.")
            with MICROMAMBA.open("wb") as output:
                shutil.copyfileobj(stream, output)
        MICROMAMBA.chmod(0o755)
    if not (ENV_ROOT / "bin" / "python").exists():
        run_command([MICROMAMBA, "create", "-y", "-p", ENV_ROOT, "-c", "conda-forge", "python=3.11", "pip"])
    ENV_PYTHON = ENV_ROOT / "bin" / "python"

if not ENV_PYTHON.exists():
    raise FileNotFoundError(f"Python environment was not created: {ENV_PYTHON}")
print("Pipeline Python:", subprocess.check_output([ENV_PYTHON, "--version"], text=True).strip())

constraints = Path("/content/cardianatomy-constraints.txt")
constraints.write_text("numpy<2\nscipy<2\n", encoding="utf-8")
run_command([ENV_PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
run_command([ENV_PYTHON, "-m", "pip", "install", "-c", constraints, "-r", UPSTREAM_ROOT / "requirements.txt"])
run_command([ENV_PYTHON, "-m", "pip", "install", "-e", str(REPO_ROOT) + "[io]"])
verification = subprocess.check_output(
    [ENV_PYTHON, "-c", "import sys, torch, nibabel, meshio, cardianatomy; print(sys.version.split()[0]); print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('cardianatomy', cardianatomy.__file__)"],
    text=True,
)
print(verification)
CUDA_AVAILABLE = subprocess.check_output([ENV_PYTHON, "-c", "import torch; print(torch.cuda.is_available())"], text=True).strip() == "True"
USE_GPU = bool(PREFER_GPU and CUDA_AVAILABLE)
if PREFER_GPU and not CUDA_AVAILABLE:
    print("WARNING: no CUDA-enabled PyTorch runtime detected. Segmentation will run on CPU and may be very slow.")
print("Segmentation GPU enabled:", USE_GPU)

## 6. Prepare the local nnU-Net workspace

The upstream segmentation stage may modify its workspace and leave temporary datasets after interruption. This notebook copies/extracts your model assets into a fresh local Colab workspace and leaves the source folder on Drive untouched. It accepts the four named tar archives or an already extracted workspace.

In [ ]:
import shutil, tarfile
from pathlib import Path

MODEL_ASSET_SOURCE = Path(MODEL_ASSET_SOURCE)
if not MODEL_ASSET_SOURCE.is_dir():
    raise FileNotFoundError(f"MODEL_ASSET_SOURCE must be a directory: {MODEL_ASSET_SOURCE}")
if "EDIT_ME" in MODEL_ASSET_SOURCE.parts:
    raise ValueError("Set MODEL_ASSET_SOURCE to your own model archive/workspace directory.")

required_archives = [
    "Dataset100_UKBB_Petersen_SAX.20240108.tar",
    "Dataset101_UKBB_LAX_2Ch.20240108.tar",
    "Dataset102_UKBB_LAX_3Ch.20240108.tar",
    "Dataset103_UKBB_LAX_4Ch.20240108.tar",
]

def safe_extract_tar(archive_path, destination):
    destination = Path(destination).resolve()
    with tarfile.open(archive_path, "r:*") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Unsafe archive member path in {archive_path.name}: {member.name}")
            if member.issym() or member.islnk():
                raise ValueError(f"Refusing link member in model archive {archive_path.name}: {member.name}")
        archive.extractall(destination)

if MODEL_WORKSPACE.exists():
    if not (MODEL_WORKSPACE / ".cardianatomy_managed_workspace").exists():
        raise RuntimeError(f"Workspace already exists and is not marked as created by this notebook: {MODEL_WORKSPACE}. Use a new RUN_ID or inspect it manually; it will not be deleted.")
else:
    MODEL_WORKSPACE.mkdir(parents=True)
    if (MODEL_ASSET_SOURCE / "nnUNet_raw").is_dir():
        print("Copying extracted workspace from Drive to local Colab storage...")
        shutil.copytree(MODEL_ASSET_SOURCE, MODEL_WORKSPACE, dirs_exist_ok=True)
    else:
        archive_paths = [MODEL_ASSET_SOURCE / name for name in required_archives]
        missing = [str(p) for p in archive_paths if not p.is_file()]
        if missing:
            raise FileNotFoundError(
                "Model source is neither an extracted workspace with nnUNet_raw nor a folder with all four required tar files. Missing: "
                + ", ".join(missing)
            )
        for archive_path in archive_paths:
            print(f"Extracting {archive_path.name} (large archive; this may take a while)...")
            safe_extract_tar(archive_path, MODEL_WORKSPACE)
    (MODEL_WORKSPACE / ".cardianatomy_managed_workspace").write_text(
        f"Created by Colab notebook on {RUN_ID}\nSource: {MODEL_ASSET_SOURCE}\n",
        encoding="utf-8",
    )

nnunet_raw = MODEL_WORKSPACE / "nnUNet_raw"
if not nnunet_raw.is_dir():
    raise FileNotFoundError(f"Expected nnUNet_raw directory is absent: {nnunet_raw}")
required_model_prefixes = [
    "Dataset100_UKBB_Petersen_SAX",
    "Dataset101_UKBB_LAX_2Ch",
    "Dataset102_UKBB_LAX_3Ch",
    "Dataset103_UKBB_LAX_4Ch",
]
for prefix in required_model_prefixes:
    matches = [p for p in nnunet_raw.iterdir() if p.is_dir() and p.name.startswith(prefix)]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one nnUNet_raw directory beginning {prefix!r}; found {[p.name for p in matches]}")
    print("Found model dataset:", matches[0].name)
print("Workspace ready:", MODEL_WORKSPACE)
print("Source model assets on Drive remain unchanged.")

## 7. Optional openCARP setup and license gate

The upstream pipeline needs openCARP for volumetric meshing and UVC/fiber generation. Current openCARP terms state that the simulator binaries are non-commercial under the Academic Public License; commercial/product-development use requires a commercial license. Review the official terms before selecting these stages: https://opencarp.org/download/license

If this run is reviewed academic non-commercial research, you may set INSTALL_OPENCARP to True to try the upstream-documented v11.0 AppImage installation. For commercial/product-development use, do not use that academic installer; provide a separately licensed commercial installation and set OPENCARP_COMMERCIAL_LICENSE_CONFIRMED = True.

This cell does not determine or grant your rights. Surface-only stages do not need openCARP.

In [ ]:
needs_opencarp = any(stage in STAGES for stage in ("volumetric", "uvc-fiber"))
if needs_opencarp:
    if not OPENCARP_TERMS_REVIEWED:
        raise RuntimeError("Volumetric/UVC-fiber is selected. Review https://opencarp.org/download/license and set OPENCARP_TERMS_REVIEWED = True only after review.")
    if INTENDED_USE == "academic_noncommercial":
        pass
    elif INTENDED_USE == "commercial_product_development":
        if not OPENCARP_COMMERCIAL_LICENSE_CONFIRMED:
            raise RuntimeError("Commercial/product-development use selected, but no commercial openCARP license is confirmed. Obtain a license or remove volumetric/UVC-fiber from STAGES.")
    else:
        raise RuntimeError("Set INTENDED_USE to the actual intended use before running openCARP-dependent stages.")
    if INSTALL_OPENCARP:
        if INTENDED_USE != "academic_noncommercial":
            raise RuntimeError("The optional AppImage installer is limited to reviewed academic non-commercial use. For commercial use, configure a separately licensed installation.")
        if not Path("/content/openCARP/usr/bin").is_dir():
            url = "https://git.opencarp.org/api/v4/projects/16/packages/generic/opencarp-appimage/v11.0/openCARP-v11.0-x86_64_AppImage.tar.gz"
            package_archive = Path("/content/openCARP-v11.0-x86_64_AppImage.tar.gz")
            print("Downloading upstream-documented openCARP v11.0 AppImage package...")
            urllib.request.urlretrieve(url, package_archive)
            package_dir = Path("/content/openCARP-appimage-package")
            package_dir.mkdir(parents=True, exist_ok=True)
            subprocess.run(["tar", "-xzf", str(package_archive), "-C", str(package_dir)], check=True)
            appimages = list(package_dir.rglob("*.AppImage"))
            if not appimages:
                raise FileNotFoundError("The openCARP archive contained no .AppImage file.")
            appimage = appimages[0]
            appimage.chmod(0o755)
            subprocess.run([str(appimage), "--appimage-extract"], cwd=package_dir, check=True)
            extracted = package_dir / "squashfs-root"
            if not (extracted / "usr" / "bin").is_dir():
                raise FileNotFoundError("Expected usr/bin in extracted openCARP AppImage.")
            shutil.move(str(extracted), "/content/openCARP")
        OPENCARP_BIN_DIR = "/content/openCARP/usr/bin"
    if not OPENCARP_BIN_DIR or not Path(OPENCARP_BIN_DIR).is_dir():
        raise FileNotFoundError("Set OPENCARP_BIN_DIR to an existing openCARP binary directory, or set INSTALL_OPENCARP = True for the reviewed academic non-commercial installation.")
    print("openCARP binaries:", OPENCARP_BIN_DIR)
    print("License review recorded:", OPENCARP_TERMS_REVIEWED, "| Intended use:", INTENDED_USE)
else:
    print("No openCARP-dependent stage selected.")

## 8. Run the selected stages

The runner writes manifests and stage logs under OUTPUT_ROOT on Drive. It rejects an existing non-empty output directory rather than overwriting results. If a run fails, preserve its folder and inspect the manifest and logs before starting a fresh run.

The progress poll reads the run manifest while the pipeline is running. The adapter writes stage records and partial artifacts as it proceeds. Keep the Colab session connected until the process completes.

In [ ]:
from pathlib import Path
import os, json, time, subprocess, shlex, shutil

DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
if OUTPUT_ROOT.exists():
    raise FileExistsError(f"Output directory already exists; results will not be overwritten: {OUTPUT_ROOT}. Rerun the configuration cell to generate a new RUN_ID.")

cmd = [
    str(ENV_PYTHON), "-m", "cardianatomy.biv_volumetric_cli",
    "--sax", str(VIEWS["SAX"]),
    "--lax-2ch", str(VIEWS["LAX_2CH"]),
    "--lax-3ch", str(VIEWS["LAX_3CH"]),
    "--lax-4ch", str(VIEWS["LAX_4CH"]),
    "--upstream-root", str(UPSTREAM_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--subject-id", CASE_ID,
    "--instance", str(INSTANCE),
    "--nnunet-workspace", str(MODEL_WORKSPACE),
    "--stages", *STAGES,
    "--python-executable", str(ENV_PYTHON),
    "--timeout-seconds", str(STAGE_TIMEOUT_SECONDS),
    "--minimum-points", str(MINIMUM_POINTS),
    "--minimum-tetrahedra", str(MINIMUM_TETRAHEDRA),
    "--minimum-shape-quality", str(MINIMUM_TETRA_SHAPE_QUALITY),
]
if TIMEFRAME is not None:
    cmd.extend(["--timeframe", str(TIMEFRAME)])
if USE_GPU:
    cmd.append("--gpu")
if needs_opencarp:
    cmd.extend(["--carp-bin-dir", str(OPENCARP_BIN_DIR), "--allow-restricted-runtime"])

LAUNCHER_LOG = Path("/content") / f"cardianatomy-launcher-{RUN_ID}.log"
print("Output will be persisted in:", OUTPUT_ROOT)
print("Command:", shlex.join(cmd))
print("Starting upstream stages.")

start = time.monotonic()
last_report = 0.0
with LAUNCHER_LOG.open("w", encoding="utf-8") as launch_log:
    process = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=launch_log, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
    while process.poll() is None:
        time.sleep(20)
        elapsed = int(time.monotonic() - start)
        if time.monotonic() - last_report >= 60:
            last_report = time.monotonic()
            manifest_path = OUTPUT_ROOT / "run-manifest.json"
            if manifest_path.is_file():
                try:
                    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
                    completed = [stage.get("name") for stage in manifest.get("stages", []) if stage.get("status") == "completed"]
                    print(f"[{elapsed//60} min] status={manifest.get('status')} current_stage={manifest.get('current_stage')} completed={completed}")
                except Exception as exc:
                    print(f"[{elapsed//60} min] Manifest updating ({type(exc).__name__}).")
            else:
                print(f"[{elapsed//60} min] Waiting for initial manifest.")
    PIPELINE_EXIT_CODE = process.returncode

launcher_archive = DRIVE_RUNS_DIR / f"{CASE_ID}_{RUN_ID}_launcher.log"
shutil.copy2(LAUNCHER_LOG, launcher_archive)
print("Exit code:", PIPELINE_EXIT_CODE)
print("Launcher log copied to:", launcher_archive)
print("Launcher log tail:")
print("\n".join(LAUNCHER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-100:]))
print("Elapsed minutes:", round((time.monotonic() - start) / 60, 2))
if PIPELINE_EXIT_CODE != 0:
    print("Pipeline did not complete successfully. Keep OUTPUT_ROOT and inspect the manifest/logs before retrying.")

## 9. Inspect the manifest and mesh QC

This summary reports the recorded execution only. It does not validate segmentation labels, registration, cardiac anatomy, UVC fields, fibers or clinical performance.

In [ ]:
from pathlib import Path
import json

manifest_path = OUTPUT_ROOT / "run-manifest.json"
if not manifest_path.is_file():
    print("No run manifest was created. Keep the launcher log and inspect installation/preflight errors.")
else:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("Overall status:", manifest.get("status"))
    print("Upstream commit:", manifest.get("upstream", {}).get("commit_sha"))
    print("Selected stages:", manifest.get("selected_stages"))
    print("Checks:")
    print(json.dumps(manifest.get("checks", {}), indent=2))
    print("\nStage records:")
    for stage in manifest.get("stages", []):
        print("-", stage.get("name"), "|", stage.get("status"), "|", stage.get("error", ""))
    accepted = manifest.get("geometry_qc_passed_volume_mesh")
    if accepted:
        print("\nQC-passed volume mesh candidate:")
        print(json.dumps({
            "path": accepted.get("path"),
            "sha256": accepted.get("sha256"),
            "point_count": accepted.get("point_count"),
            "tetra_count": accepted.get("tetra_count"),
            "qc": accepted.get("qc"),
        }, indent=2))
    else:
        candidates = manifest.get("volume_mesh_candidates", [])
        print("\nNo QC-passed volume mesh was recorded. Candidate count:", len(candidates))
        for candidate in candidates[:10]:
            print(candidate.get("path"), "points=", candidate.get("point_count"),
                  "tetra=", candidate.get("tetra_count"),
                  "qc_passed=", candidate.get("qc", {}).get("passed"))
    print("\nPersisted outputs:", OUTPUT_ROOT)
    print("Review outputs against source images before scientific use.")

## 10. What to retain and what this does not prove

Retain the complete run folder in Drive, especially preflight-manifest.json, run-manifest.json, stage logs, input hashes, the upstream commit SHA, the produced geometry and its QC record.

Do not delete a failed run until its partial outputs have been inspected. For repeat attempts, rerun the configuration cell to generate a new timestamped output folder.

A successful command establishes only that the selected software produced recorded artifacts and that the adapter's configured QC checks passed. Independently review image/label overlays, cross-view alignment, segmentation quality, chamber boundaries, mesh topology, coordinate conventions and suitability for any downstream solver. This is a research workflow, not a clinical tool.